# Example 15.6 / 2: Compare a General Chatbot with AMPLbot

Companion notebook for **15.6 Talks on AI for Optimization** in the AMPL lab Documentation, section "Compare a General Chatbot with AMPLbot". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Compare a General Chatbot with AMPLbot

The general chatbot returned a model, a data file and a run file; running them produced an error
in the part of the model that uses ordered nodes. AMPLbot returned an AMPL model with a Python
script that loaded the data and solved; it ran without changes and reported an optimal tour.
He noted that a one-line prompt is weak on purpose and that stronger prompts give better results,
and that assistants also help with choosing among a solver's many options. In the same session,
Cardinal Operations presented ORLM, an LLM trained to build optimization models and call their
COPT solver, and a classroom "cookie game" in which student teams use it on business cases.

Whatever assistant answers this prompt, check the tour it returns, not only the solve status. A
traveling salesman model of the kind to expect, with Miller-Tucker-Zemlin ordering constraints
against subtours (Cuts and Subtour Elimination (AMPL lab documentation, section 12.3)), and the check:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set CITIES ordered;                                  # cities; the first is the start
param x {CITIES};                                    # coordinates
param y {CITIES};
param dist {i in CITIES, j in CITIES} = sqrt((x[i] - x[j])^2 + (y[i] - y[j])^2);
param n = card(CITIES);

var Go {i in CITIES, j in CITIES: i <> j} binary;    # 1 if the tour goes from i to j
var Order {CITIES} >= 1, <= n;                       # position of each city in the tour

minimize Length: sum {i in CITIES, j in CITIES: i <> j} dist[i,j] * Go[i,j];

subject to Leave {i in CITIES}: sum {j in CITIES: j <> i} Go[i,j] = 1;  # one successor
subject to Enter {j in CITIES}: sum {i in CITIES: i <> j} Go[i,j] = 1;  # one predecessor
subject to Start: Order[first(CITIES)] = 1;                             # tour starts at the first city
# Miller-Tucker-Zemlin: going from i to j puts j later than i, which rules out subtours
subject to No_Subtour {i in CITIES, j in CITIES: i <> j and j <> first(CITIES)}:
    Order[i] - Order[j] + n * Go[i,j] <= n - 1;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["CITIES"] = ["Depot", "B", "C", "D", "E"]
ampl.param["x"] = {"Depot": 0, "B": 4, "C": 4, "D": 0, "E": 2}
ampl.param["y"] = {"Depot": 0, "B": 0, "C": 3, "D": 3, "E": 5}

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result, round(ampl.obj["Length"].value(), 4))
succ = {i: j for (i, j), v in ampl.var["Go"].to_dict().items() if v > 0.5}
tour, city = ["Depot"], succ["Depot"]
while city != "Depot":
    tour.append(city)
    city = succ[city]
print("tour:", tour, "visits all cities:", len(tour) == 5)

HiGHS 1.14.0: optimal solution; objective 15.65685425
26 simplex iterations
1 branching nodes
solved 15.6569
tour: ['Depot', 'B', 'C', 'E', 'D'] visits all cities: True


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set CITIES := Depot B C D E;
param:   x  y :=
Depot    0  0
B        4  0
C        4  3
D        0  3
E        2  5;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Length, {i in CITIES, j in CITIES: i <> j and Go[i,j] > 0.5};

HiGHS 1.14.0: optimal solution; objective 15.65685425
26 simplex iterations
1 branching nodes
Length = 15.6569

set {i in CITIES, j in CITIES: i != j && Go[i,j] > 0.5}  :=
(Depot,B)   (B,C)       (C,E)       (D,Depot)   (E,D);

